In [ ]:
import pandas as pd
df_feature = pd.read_csv("instance_summary_CS.csv")
df_result = pd.read_csv("AR_asset_22-38.csv")
df_result["instance"] = df_result["instance"].astype(int)
df_result.drop(columns=["Asset","start","end","Unnamed: 0"], inplace=True)
df_result.dropna(inplace=True)
# df_result.rename(columns={"instance #": "instance"}, inplace=True)

merged_df = pd.merge(df_feature, df_result, on='instance', how='inner')
merged_df.drop(columns=["Unnamed: 0","Unnamed: 0.1","start_year"], inplace=True)
merged_df.to_csv("merged_CS_AR.csv",index=False)

In [2]:
print(merged_df.columns)
print(merged_df.shape)


Index(['instance', 'P/E mean', 'P/E standard deviation', 'P/B mean',
       'P/B standard deviation', 'P/S mean', 'P/S standard deviation',
       'P/CF mean', 'P/CF standard deviation', 'EV/EBITDA mean',
       'EV/EBITDA standard deviation', 'Gross Margin mean',
       'Gross Margin standard deviation', 'Net Margin mean',
       'Net Margin standard deviation', 'ROA mean', 'ROA standard deviation',
       'ROE mean', 'ROE standard deviation', 'FCF mean',
       'FCF standard deviation', 'FCF per Share mean',
       'FCF per Share standard deviation', 'SPS mean',
       'SPS standard deviation', 'PVS mean', 'PVS standard deviation',
       'RAPS mean', 'RAPS standard deviation', 'EVC mean',
       'EVC standard deviation', 'VEC mean', 'VEC standard deviation',
       'PLF mean', 'PLF standard deviation', 'IQS mean',
       'IQS standard deviation', 'Period', 'EW', 'MCW', 'MVO', 'MS-M', 'OH-M',
       'MS-T', 'OH-T', 'MS-L', 'OH-L'],
      dtype='object')
(42, 47)


In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# feature and target columns
feature_cols = merged_df.columns[1:38]
target_cols = merged_df.columns[38:]

X = merged_df[feature_cols]

# 保存每个算法的预测和实际
all_actual = pd.DataFrame(index=merged_df.index)
all_pred = pd.DataFrame(index=merged_df.index)

# 保存 test instance id
_, X_test, _, _ = train_test_split(X, merged_df[target_cols[0]], test_size=0.1, random_state=42)
test_indices = X_test.index

for target in target_cols:
    y = merged_df[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # 存储结果
    all_actual.loc[y_test.index, target] = y_test
    all_pred.loc[y_test.index, target] = y_pred

# 只保留 test 部分
actual_test = all_actual.loc[test_indices]
pred_test = all_pred.loc[test_indices]

# 计算 SBS, VBS, Selection Result
result_summary = pd.DataFrame(index=test_indices)
result_summary["Instance"] = merged_df.loc[test_indices, "instance"].values

# SBS: OH-L 实际值
result_summary["SBS"] = actual_test["OH-L"]

# VBS: 所有算法中实际分数的最大值
result_summary["VBS"] = actual_test.max(axis=1)

# Selection Result: 根据预测分数选择算法，然后取该算法的 actual
best_pred_algo = pred_test.idxmax(axis=1)  # 每行预测最高的算法名
selection_result = []
for i, algo in best_pred_algo.items():
    selection_result.append(actual_test.loc[i, algo])

result_summary["Selection Result"] = selection_result

# === 统计结果 ===
stats = pd.DataFrame({
    "Mean": result_summary[["SBS", "VBS", "Selection Result"]].mean(),
    "Std": result_summary[["SBS", "VBS", "Selection Result"]].std(),
    "Min": result_summary[["SBS", "VBS", "Selection Result"]].min(),
    "Max": result_summary[["SBS", "VBS", "Selection Result"]].max(),
}).T

print("=== 每个测试样本的结果 ===")
print(result_summary.head())

print("\n=== 总体统计 ===")
print(stats)

# 保存
result_summary.to_csv("AR_strategy_selection.csv", index=False)
stats.to_csv("AR_strategy_selection_stats.csv")


=== 每个测试样本的结果 ===
     Instance       SBS       VBS  Selection Result
340       341  0.887976  1.013040          0.801387
290       291  0.684541  0.684541          0.407039
54         55  0.784251  0.820039          0.820039
198       199 -0.075171  0.106076         -0.002922
453       454 -0.075752  0.132871          0.116028

=== 总体统计 ===
           SBS       VBS  Selection Result
Mean  0.275439  0.425923          0.370699
Std   0.501782  0.501870          0.479549
Min  -0.329056 -0.109768         -0.298258
Max   3.321801  3.321801          3.296158
